# Liu2024 — Per-Subject Decodability (Permutation Test) + Subgroup Analysis

The faithful FgMDM reproduction showed the cohort is **bimodal**: every fixed band sits at
chance *on average*, the "oracle" best-band number (63%) is largely selection-on-noise, yet a
handful of subjects decode far above chance. This notebook answers the question that actually
changes the project: **which subjects carry real left-vs-right MI signal?** — so all future
S-JEPA-vs-Riemannian comparisons run on that subgroup instead of being washed out.

**Method**
- Reuses the *identical* covariance features from the faithful reproduction (native 500 Hz,
  0–4 s from trigger, SCM covariance, the 8 Liu bands).
- **Honest decodability test:** on a fixed broadband **8–30 Hz** (no band cherry-picking), a
  log-Euclidean nearest-Riemannian-mean classifier under repeated stratified CV; a
  **label-permutation null** gives a per-subject p-value; **Benjamini–Hochberg FDR** and
  Bonferroni control the 50-subject multiplicity.
- **Reported alongside:** the affine-invariant **FgMDM** accuracy (the paper's classifier), no
  permutation needed.
- **Oracle correction:** the best-of-8-bands statistic gets a **selection-aware** null (max over
  bands on permuted labels), so its corrected p-value reveals how much of the oracle accuracy
  was real vs. manufactured by picking the best band.
- **Optional:** point `CONFIG["decode"]["sjepa_subject_metrics_path"]` at a prior S-JEPA run's
  per-subject metrics to get a paired comparison on the decodable subgroup.

# 1. Setup (carried verbatim from the faithful reproduction)

In [7]:
import os, re, json, hashlib, random, builtins, platform
import sys
import torch
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy import signal
from scipy.linalg import eigh

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as exc:
    HAVE_MPL = False
    print(f"[setup] matplotlib unavailable -> plots skipped: {exc}")

try:
    from pyriemann.estimation import Covariances
    from pyriemann.classification import FgMDM, MDM
    HAVE_PYRIEMANN = True
except Exception as exc:
    HAVE_PYRIEMANN = False
    print(f"[setup] pyriemann unavailable -> FALLBACK to log-Euclidean MDM (NOT faithful FgMDM): {exc}")

print("MDM backend:",
      "pyriemann.FgMDM (faithful DGFMDRM)" if HAVE_PYRIEMANN else "log-Euclidean MDM fallback (no DGF)")


[2026-06-14 11:09:41] MDM backend: pyriemann.FgMDM (faithful DGFMDRM)


## 1.1 Channel + source constants (carried)

In [8]:
# Liu2024 source MAT channel conventions.
# Source files are organized as trials x 33 channels x samples:
#   0..29 = EEG-like channels, index 17 = CPz source reference,
#   30..31 = EOG, 32 = marker.
#
# The channel labels below follow the Liu2024 paper / EEGLAB location files.
# This matters for montage-dependent topomaps and any channel-position metadata.
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]

# EOG and marker channels available in Liu2024 source MAT files.
SOURCE_EOG_CHANNEL_INDICES = [30, 31]
SOURCE_MARKER_CHANNEL_INDEX = 32


## 1.2 CONFIG (carried) — includes the `liu` block used to build the features

In [9]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-sjepa-prelocal"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "baseline_sjepa_prelocal",
    "config_note": "Clean MNE-style preprocessing pipeline builder + S-JEPA hyperparameter controls.",

    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    "subjects_to_use": None,
    "exclude_subjects": [],
    "source_unit": "microvolts",
    "final_model_unit": "microvolts",

    # ------------------------------------------------------------------
    # Source-domain preprocessing before creating MNE RawArray
    # ------------------------------------------------------------------
    "demean_mode": "none",      # none, trial_mean, baseline_window_mean
    "baseline_window_s": [0.0, 2.0],
    "detrend_mode": "none",                     # none, constant, linear
    "eog_correction": "none",                   # none, linear_regression

    # Robust source-domain clipping / winsorization. Use cautiously.
    "artifact_clip_mode": "none",               # none, absolute, percentile
    "artifact_clip_abs_value": None,             # in source_unit, e.g. 150.0 when source_unit=microvolts
    "artifact_clip_percentile": 99.5,

    # ------------------------------------------------------------------
    # MNE Raw-level preprocessing
    # ------------------------------------------------------------------
    "reference_mode": "average",                # average, none
    "reference_timing": "before_resample_filter",  # before_resample_filter, after_resample_before_filter, after_filter
    "resample": True,
    "resample_sfreq": 128,

    "filter_enabled": True,
    "filter_low": 0.5,
    "filter_high": 40.0,
    "filter_method": "fir",                     # fir, iir
    "filter_phase": "zero",                     # zero, zero-double, minimum (FIR only)
    "filter_fir_design": "firwin",              # firwin, firwin2 (FIR only)
    "filter_l_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_h_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_iir_params": None,                    # example: {"order": 2, "ftype": "butter"}

    "notch_freqs": None,                          # example: [50.0]
    "notch_before_bandpass": False,

    # ------------------------------------------------------------------
    # Windowing and post-window cleaning
    # ------------------------------------------------------------------
    "target_window_s": 4.2,
    "target_window_samples": 537,
    "mi_window_start_s": 1.5,

    "reject_bad_trials": False,
    "reject_peak_to_peak_threshold": None,       # in final_model_unit
    "reject_abs_threshold": None,                # in final_model_unit
    "min_trials_per_class_after_reject": None,

    # ------------------------------------------------------------------
    # Fold-safe normalization. train_* modes are fit on each training split only.
    # ------------------------------------------------------------------
    "normalization_mode": "none",               # none, train_global_zscore, train_channel_zscore, train_channel_robust, trial_global_zscore, trial_channel_zscore
    "normalization_eps": 1e-6,

    # ------------------------------------------------------------------
    # Model / downstream strategy
    # ------------------------------------------------------------------
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": "from_pretrained",
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
    "strategy": "new",                          # new, full
    "warmup_epochs": 10,

    # ------------------------------------------------------------------
    # Evaluation protocol
    # ------------------------------------------------------------------
    "evaluation_mode": "stratified_kfold",      # stratified_kfold, liu2024_repeated_60_40, repeated_stratified_split
    "cv_folds": 5,
    "n_repeats": 10,
    "test_size": 0.4,
    "split_random_state": 2026,
    "assert_balanced_folds": True,

    # ------------------------------------------------------------------
    # Training hyperparameters
    # ------------------------------------------------------------------
    "batch_size": 4,
    "n_epochs": 5000,
    "early_stopping_patience": 50,
    "val_split": 0.2,
    "learning_rate": 0.0003,
    "optimizer_name": "adam",                   # adam, adamw
    "weight_decay": 0.0,
    "gradient_clip_norm": None,
    "checkpoint_metric": "valid_loss",           # valid_loss, valid_balanced_accuracy
    "label_smoothing": 0.0,
    "prediction_balance_loss_weight": 1.0,

    # ------------------------------------------------------------------
    # braindecode on-the-fly augmentation.
    # Applied to the TRAINING iterator ONLY (via AugmentedDataLoader), so the
    # validation split skorch carves out internally is never augmented -> no leakage.
    # There is no fixed "number of augmented samples": the model sees a freshly
    # augmented view of the train fold every epoch. Control INTENSITY with each
    # transform's "probability" (how often it fires) and its magnitude params.
    # Ready-to-use configs are in the markdown cell just below CONFIG.
    # ------------------------------------------------------------------
    # "augmentation": {
    #     "enabled": False,        # master switch
    #     "name": "none",          # label, saved with artifacts
    #     "random_state": 2026,
    #     # each entry: {"name": <transform>, "probability": 0..1, <transform params>}
    #     "transforms": [],
    # },

    "augmentation": {
        "enabled": True,
        "name": "time_mask",
        "random_state": 2026,
        "transforms": [
        {
            "mask_len_samples": 64,
            "name": "smooth_time_mask",
            "probability": 0.5
        }
        ]
    },

    # ------------------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------------------
    "seed": 2026,
    "set_seed": True,
    "cv_random_state": 2026,
    "val_split_random_state": 2026,

    # ------------------------------------------------------------------
    # Diagnostics / interpretation
    # ------------------------------------------------------------------
    "extract_spatial_conv_weights": True,
    "save_spatial_weight_plots": False,
    "plot_individual_spatial_filters": False,
    "max_spatial_filters_to_plot": 8,
    "topomap_dpi": 160,
    "topomap_value_mode": "relative_zscore",    # raw, relative_zscore, relative_percent
    "topomap_cmap": "RdBu_r",
    "collapse_threshold": 0.9,
    "log_spatial_update_stats": True,
    "log_probability_diagnostics": True,
}

# ------------------------------------------------------------------ #
#  Faithful Liu2024 TWFB+DGFMDRM reproduction settings.              #
# ------------------------------------------------------------------ #
CONFIG["experiment_name"] = "twfb_dgfmdm_faithful_reproduction"
CONFIG["artifact_dir"] = str(WORKING_DIR / "artifacts" / "liu2024-twfb-dgfmdm-faithful")
CONFIG["config_note"] = "Direct port of TWFB_DGFMDM.m: native 500 Hz, 0-4 s from trigger, SCM cov, FgMDM, 8 bands, 24/16 x10."

CONFIG["liu"] = {
    # --- channels: MATLAB channel = [1:17 19:30] (1-based) -> drop the reference (idx 17, 0-based) ---
    "keep_channel_indices": list(range(0, 17)) + list(range(18, 30)),   # 29 EEG channels, 0-based
    "marker_channel_index": 32,           # MATLAB column 33
    "trigger_value": 2,                   # MATLAB find(col33==2)
    "sfreq_hz": 500,                      # native; no resample
    # --- window: 0..4 s after trigger (MATLAB keeps samples 801:2800 of the trigger-aligned grab) ---
    "window_start_after_trigger_s": 0.0,
    "window_length_s": 4.0,
    "fallback_cue_sample": 750,           # used only if a trial has no detectable trigger (1.5 s * 500)
    # --- filtering ---
    "notch_hz": 50.0,
    "filter_order": 4,
    "freq_bands_hz": [[8, 12], [8, 20], [8, 30], [12, 20], [15, 20], [15, 30], [20, 30], [8, 15]],
    # --- classifier / metric ---
    "metric": "riemann",
    "cov_estimator": "scm",               # SCM == X^T X (matches MATLAB SS'*SS up to scale)
    "cov_shrinkage_fallback": 1e-3,       # only for the log-Euclidean fallback
    # --- protocol (MATLAB: 10 reps, 24 train / 16 test, max over bands) ---
    "n_repeats": 10,
    "n_train": 24,
    "n_test": 16,
    "stratified_split": False,            # MATLAB uses plain randperm; set True for a balanced variant
    "nested_inner_folds": 3,              # for the unbiased band-selection number
    "max_subjects": None,
    "base_split_seed": 2026,
}
print(f"Experiment: {CONFIG['experiment_name']}")
print(f"bands: {CONFIG['liu']['freq_bands_hz']}")
print(f"protocol: {CONFIG['liu']['n_repeats']}x {CONFIG['liu']['n_train']}/{CONFIG['liu']['n_test']} split")


[2026-06-14 11:09:41] Experiment: twfb_dgfmdm_faithful_reproduction
[2026-06-14 11:09:41] bands: [[8, 12], [8, 20], [8, 30], [12, 20], [15, 20], [15, 30], [20, 30], [8, 15]]
[2026-06-14 11:09:41] protocol: 10x 24/16 split


## 1.3 Decodability-analysis settings

In [10]:
# Override identity for this analysis; keep the carried `liu` feature settings intact.
CONFIG["experiment_name"] = "decodability_permutation_subgroup"
CONFIG["artifact_dir"] = str(WORKING_DIR / "artifacts" / "liu2024-decodability-permutation")
CONFIG["config_note"] = "Per-subject label-permutation decodability test on Liu-faithful covariance features."

CONFIG["decode"] = {
    "broadband": "8-30",        # fixed band for the honest test (one of the carried liu bands)
    "cv_folds": 5,
    "cv_repeats": 2,
    "n_permutations": 1000,
    "alpha": 0.05,
    "seed": 2026,
    "report_fgmdm": True,                 # also compute affine-invariant FgMDM CV (cheap, no perm)
    "oracle_selection_null": True,        # selection-aware null over all 8 bands
    "sjepa_subject_metrics_path": None,   # CSV/JSON of per-subject S-JEPA balanced accuracy
    "max_subjects": None,
}
print("Decodability config:", CONFIG["decode"])


[2026-06-14 11:09:42] Decodability config: {'broadband': '8-30', 'cv_folds': 5, 'cv_repeats': 2, 'n_permutations': 1000, 'alpha': 0.05, 'seed': 2026, 'report_fgmdm': True, 'oracle_selection_null': True, 'sjepa_subject_metrics_path': None, 'max_subjects': None}


## 1.4 Derived constants / artifacts / reproducibility (carried)

In [11]:
# Liu2024 source MAT constants.
LIU_SOURCE_SFREQ = 500
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
LIU_EXPECTED_SOURCE_CHANNELS = 33
LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL = 4000

# Liu source MAT files include EEG + EOG + marker channels.
# Keep the 29 EEG channels used in the Liu paper baseline and drop CPz because it is the source reference channel.
EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [
    name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
    if idx != SOURCE_REFERENCE_INDEX
]

SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
TARGET_N_CLASSES = 2

if bool(CONFIG.get("resample", True)):
    EFFECTIVE_SFREQ = float(CONFIG.get("resample_sfreq", 128))
else:
    EFFECTIVE_SFREQ = float(LIU_SOURCE_SFREQ)

CONFIG["effective_sfreq"] = EFFECTIVE_SFREQ
CONFIG["sfreq"] = EFFECTIVE_SFREQ  # compatibility with existing cells/artifacts

if CONFIG.get("target_window_samples", None) is None:
    WINDOW_SAMPLES = int(round(float(CONFIG["target_window_s"]) * EFFECTIVE_SFREQ))
else:
    WINDOW_SAMPLES = int(CONFIG["target_window_samples"])

TARGET_TRIAL_DURATION_S = WINDOW_SAMPLES / EFFECTIVE_SFREQ
MI_WINDOW_START_SAMPLE = int(round(float(CONFIG["mi_window_start_s"]) * EFFECTIVE_SFREQ))
MI_WINDOW_STOP_SAMPLE = MI_WINDOW_START_SAMPLE + WINDOW_SAMPLES

PREPROCESSING_KEYS = [
    "source_unit", "final_model_unit",
    "demean_mode", "baseline_window_s", "detrend_mode", "eog_correction",
    "artifact_clip_mode", "artifact_clip_abs_value", "artifact_clip_percentile",
    "reference_mode", "reference_timing", "resample", "resample_sfreq", "effective_sfreq",
    "filter_enabled", "filter_low", "filter_high", "filter_method", "filter_phase",
    "filter_fir_design", "filter_l_trans_bandwidth", "filter_h_trans_bandwidth", "filter_iir_params",
    "notch_freqs", "notch_before_bandpass",
    "mi_window_start_s", "target_window_s", "target_window_samples",
    "reject_bad_trials", "reject_peak_to_peak_threshold", "reject_abs_threshold",
    "normalization_mode", "normalization_eps",
]

TRAINING_KEYS = [
    "strategy", "batch_size", "learning_rate", "optimizer_name", "weight_decay",
    "val_split", "early_stopping_patience", "n_epochs",
    "augmentation",
]

EVALUATION_KEYS = [
    "evaluation_mode", "cv_folds", "n_repeats", "test_size",
    "cv_random_state", "split_random_state", "val_split_random_state",
]

def summarize_selected_config(keys):
    return {k: CONFIG.get(k) for k in keys}

PREPROCESSING_CONFIG = summarize_selected_config(PREPROCESSING_KEYS)
TRAINING_CONFIG = summarize_selected_config(TRAINING_KEYS)
EVALUATION_CONFIG = summarize_selected_config(EVALUATION_KEYS)

def print_config_block(title, values):
    print(title)
    for key, value in values.items():
        print(f"  {key:34s}: {value}")

print("Effective Liu2024 Source MAT settings:")
print(f"  Experiment:                        {CONFIG.get('experiment_name')}")
print(f"  Note:                              {CONFIG.get('config_note')}")
print(f"  Channels:                          {len(EEG_CHANNEL_NAMES)}")
print(f"  Channel names:                     {EEG_CHANNEL_NAMES}")
print(f"  Source sfreq:                      {LIU_SOURCE_SFREQ} Hz")
print(f"  Effective sfreq:                   {EFFECTIVE_SFREQ} Hz")
print(f"  MI window start / samples:         {CONFIG['mi_window_start_s']} s / {WINDOW_SAMPLES}")
print(f"  Effective window duration:         {TARGET_TRIAL_DURATION_S:.4f} s")
print(f"  Evaluation mode:                   {CONFIG.get('evaluation_mode')}")
print(f"  Fixed seed:                        base={CONFIG.get('seed')} | cv={CONFIG.get('cv_random_state')} | split={CONFIG.get('split_random_state')} | val={CONFIG.get('val_split_random_state')}")
print_config_block("\nPreprocessing config:", PREPROCESSING_CONFIG)
print_config_block("\nTraining config:", TRAINING_CONFIG)
print_config_block("\nEvaluation config:", EVALUATION_CONFIG)


[2026-06-14 11:09:42] Effective Liu2024 Source MAT settings:
[2026-06-14 11:09:42]   Experiment:                        decodability_permutation_subgroup
[2026-06-14 11:09:42]   Note:                              Per-subject label-permutation decodability test on Liu-faithful covariance features.
[2026-06-14 11:09:42]   Channels:                          29
[2026-06-14 11:09:42]   Channel names:                     ['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'FCz', 'FC3', 'FC4', 'FT7', 'FT8', 'Cz', 'C3', 'C4', 'T3', 'T4', 'CP3', 'CP4', 'TP7', 'TP8', 'Pz', 'P3', 'P4', 'T5', 'T6', 'Oz', 'O1', 'O2']
[2026-06-14 11:09:42]   Source sfreq:                      500 Hz
[2026-06-14 11:09:42]   Effective sfreq:                   128.0 Hz
[2026-06-14 11:09:42]   MI window start / samples:         1.5 s / 537
[2026-06-14 11:09:42]   Effective window duration:         4.1953 s
[2026-06-14 11:09:42]   Evaluation mode:                   stratified_kfold
[2026-06-14 11:09:42]   Fixed seed:           

In [12]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass

    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")


[2026-06-14 11:09:42] Run ID:     20260614_1109_9d3f1b1b
[2026-06-14 11:09:42] Artifacts:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-decodability-permutation/20260614_1109_9d3f1b1b
[2026-06-14 11:09:42] Config:     /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-decodability-permutation/20260614_1109_9d3f1b1b/config.json


In [13]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")


[2026-06-14 11:09:42] Using device: mps
[2026-06-14 11:09:42] Seed initialized: 2026


## 1.5 Data-loading helpers (carried)

In [14]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    s = str(path)
    m = re.search(r"sub[-_ ]?(\d{1,2})", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from path: {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    """Recursively walk scipy-loaded MATLAB dicts/structs.

    Liu2024 source files may expose only a top-level `eeg` object instead of
    top-level `rawdata` and `labels`. This walker lets the loader find nested
    arrays without assuming one exact MATLAB struct layout.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")

def mat_structure_preview(path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({
                "name": name,
                "type": "ndarray",
                "shape": str(value.shape),
                "dtype": str(value.dtype),
            })
        else:
            rows.append({
                "name": name,
                "type": type(value).__name__,
                "shape": "",
                "dtype": "",
            })
    return pd.DataFrame(rows).head(max_rows)

def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")

    # Prefer the label-count axis as the trial axis when labels are available.
    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    # After trial-axis normalization, the time axis should be the largest axis.
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize rawdata to trials x channels x samples, got {arr.shape}")
    return arr

def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if "rawdata" in lname or "raw" in lname or "data" in lname:
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    if "eeg" in lname:
        score += 1
    return score

def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    score = 0
    if "label" in lname or "class" in lname or lname.split(".")[-1] in {"y", "labels"}:
        score += 10
    if flat.size in (39, 40):
        score += 3
    if unique and unique.issubset({"0", "1", "2"}):
        score += 2
    return score

def validate_liu_source_subject(rawdata, labels, subject_id, path=None):
    """Validate the fixed Liu source MAT layout assumptions."""
    expected_trials = LIU_EXPECTED_TRIALS_PER_SUBJECT
    expected_channels = LIU_EXPECTED_SOURCE_CHANNELS
    expected_samples = LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL

    if rawdata.shape[0] != labels.size:
        raise ValueError(
            f"Subject {subject_id}: labels/trials mismatch. "
            f"rawdata={rawdata.shape}, labels={labels.shape}, path={path}"
        )
    if rawdata.shape[0] != expected_trials:
        print(f"WARNING subject {subject_id}: expected {expected_trials} trials, got {rawdata.shape[0]}")
    if rawdata.shape[1] < len(SOURCE_EEG_CHANNEL_INDICES_30):
        raise ValueError(f"Subject {subject_id}: expected at least 30 EEG-like channels, got {rawdata.shape}")
    if rawdata.shape[1] != expected_channels:
        print(f"WARNING subject {subject_id}: expected {expected_channels} source channels, got {rawdata.shape[1]}")
    if rawdata.shape[2] != expected_samples:
        print(f"WARNING subject {subject_id}: expected {expected_samples} samples/trial, got {rawdata.shape[2]}")

    unique = set(np.unique(labels).astype(int).tolist())
    if not unique.issubset({0, 1, 2}):
        raise ValueError(f"Subject {subject_id}: unexpected labels {sorted(unique)}")

    y0 = labels_to_zero_based(labels)
    counts = np.bincount(y0, minlength=TARGET_N_CLASSES)
    if counts.min() == 0:
        raise ValueError(f"Subject {subject_id}: missing class after zero-based conversion, counts={counts.tolist()}")
    if counts[0] != counts[1]:
        print(f"WARNING subject {subject_id}: class counts are not balanced: {counts.tolist()}")

def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)

    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates, label_candidates = [], []
    for name, arr in arrays:
        if arr.ndim == 3:
            raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
        elif arr.ndim in (1, 2):
            label_candidates.append((_score_label_candidate(name, arr), name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        preview_path = ARTIFACT_DIR / f"mat_structure_failure_{Path(path).stem}.csv"
        preview.to_csv(preview_path, index=False)
        print(f"MAT structure preview for failure saved to: {preview_path}")
        display(preview.head(40))
        raise KeyError(f"Could not locate 3D raw data and labels in {path}")

    _, raw_name, raw_arr = sorted(raw_candidates, key=lambda x: x[0], reverse=True)[0]
    _, label_name, label_arr = sorted(label_candidates, key=lambda x: x[0], reverse=True)[0]

    labels = np.asarray(label_arr).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(raw_arr, labels=labels).astype(np.float64)

    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label count mismatch in {path}: labels={labels.shape}, rawdata={rawdata.shape}")

    return rawdata, labels.astype(int), raw_name, label_name


## 2. Liu-faithful windowing / filtering / covariance (carried — identical features)

In [15]:
LIU = CONFIG["liu"]
FS = float(LIU["sfreq_hz"])
WIN_LEN = int(round(LIU["window_length_s"] * FS))                 # 2000 samples
WIN_START = int(round(LIU["window_start_after_trigger_s"] * FS))  # 0
KEEP = np.array(LIU["keep_channel_indices"], dtype=int)

def detect_trigger(trial_all_channels):
    """Return the 0-4 s window start sample for one trial (trials are 33 x 4000 here)."""
    midx = LIU["marker_channel_index"]
    if trial_all_channels.shape[0] > midx:
        marker = np.rint(trial_all_channels[midx]).astype(int)
        hits = np.where(marker == int(LIU["trigger_value"]))[0]
        if hits.size:
            return int(hits[0]) + WIN_START, True
    return int(LIU["fallback_cue_sample"]) + WIN_START, False

def _notch(x, fs, f0, q=30.0):
    b, a = signal.iirnotch(f0 / (fs / 2.0), q)
    return signal.filtfilt(b, a, x, axis=-1)

def _bandpass(x, lo, hi, fs, order):
    nyq = 0.5 * fs
    b, a = signal.butter(order, [max(lo / nyq, 1e-4), min(hi / nyq, 0.999)], btype="band")
    return signal.filtfilt(b, a, x, axis=-1)

def subject_band_windows(raw_trials):
    """raw_trials: (n_trials, 33, 4000). Returns dict band_label -> (n_trials, 29, WIN_LEN) and trigger info."""
    n_tr = raw_trials.shape[0]
    starts, detected = [], 0
    for i in range(n_tr):
        s, ok = detect_trigger(raw_trials[i])
        # clamp so the window fits inside the epoch
        s = min(max(s, 0), raw_trials.shape[2] - WIN_LEN)
        starts.append(s); detected += int(ok)
    out = {}
    for lo, hi in LIU["freq_bands_hz"]:
        blab = f"{lo}-{hi}"
        W = np.empty((n_tr, KEEP.size, WIN_LEN))
        for i in range(n_tr):
            x = raw_trials[i, KEEP, :]                       # (29, 4000)
            x = _notch(x, FS, LIU["notch_hz"])               # 50 Hz notch on full epoch
            x = _bandpass(x, lo, hi, FS, LIU["filter_order"])# band-pass on full epoch (clean edges)
            W[i] = x[:, starts[i]:starts[i] + WIN_LEN]       # crop 0-4 s
        out[blab] = W
    return out, detected, n_tr

def covariances(windows):
    """(n_trials, 29, T) -> (n_trials, 29, 29). pyriemann SCM if available else numpy SCM."""
    if HAVE_PYRIEMANN:
        return Covariances(estimator=LIU["cov_estimator"]).transform(windows)
    n_tr, n_ch, T = windows.shape
    C = np.empty((n_tr, n_ch, n_ch))
    for i in range(n_tr):
        Xi = windows[i]
        C[i] = Xi @ Xi.T                                      # SCM == X X^T (matches SS'*SS)
    return C

def labels_zero_based(labels):
    """MATLAB labels are 1 (left) / 2 (right). Map to 0/1; class 1 == right hand."""
    return np.asarray(labels, dtype=int) - 1


## 2.1 Classifier helpers (carried — `_logm_spd`, FgMDM path)

In [16]:
def _logm_spd(C):
    w, V = eigh(C); w = np.clip(w, 1e-12, None)
    return (V * np.log(w)) @ V.T

def _logeuclid_mdm(cov_tr, y_tr, cov_te, n_classes, shrink):
    # regularize toward scaled identity, take matrix logs, classify by nearest mean-of-logs
    def reg(C):
        n = C.shape[0]; tr = np.trace(C) / n
        return (1 - shrink) * C + shrink * tr * np.eye(n)
    logs_tr = np.stack([_logm_spd(reg(C)) for C in cov_tr])
    logs_te = np.stack([_logm_spd(reg(C)) for C in cov_te])
    means = []
    for c in range(n_classes):
        m = (y_tr == c)
        means.append(logs_tr[m].mean(axis=0) if m.any() else np.zeros_like(logs_tr[0]))
    means = np.stack(means)
    d = np.stack([np.linalg.norm(logs_te - mu, axis=(1, 2)) for mu in means], axis=1)
    return d.argmin(axis=1).astype(int)

def classify_fold(cov_tr, y_tr, cov_te, n_classes=2):
    if HAVE_PYRIEMANN:
        clf = FgMDM(metric=LIU["metric"])
        clf.fit(cov_tr, y_tr)
        return np.asarray(clf.predict(cov_te), dtype=int)
    return _logeuclid_mdm(cov_tr, y_tr, cov_te, n_classes, LIU["cov_shrinkage_fallback"])


## 3. Load raw subjects (carried)

In [17]:
SRC = Path(CONFIG["source_extract_dir"])
mat_files = find_source_mat_files(SRC)
print(f"Found {len(mat_files)} .mat files under {SRC}")
if not mat_files:
    raise FileNotFoundError(f"No .mat files under {SRC}. Set CONFIG['source_extract_dir'].")

RAW_SUBJECTS = {}
for path in mat_files:
    sid = subject_id_from_path(path)
    rawdata, labels, _, _ = load_subject_mat(path)     # (trials, 33, 4000), labels in {1,2}
    RAW_SUBJECTS[str(sid)] = (rawdata, labels_zero_based(labels))
SUBJECTS = sorted(RAW_SUBJECTS.keys(), key=lambda s: int(s))
if LIU["max_subjects"]:
    SUBJECTS = SUBJECTS[:int(LIU["max_subjects"])]
print(f"Loaded {len(SUBJECTS)} subjects. Example raw shape: {RAW_SUBJECTS[SUBJECTS[0]][0].shape}")
print(f"Window: {WIN_LEN} samples = {LIU['window_length_s']}s @ {FS:.0f} Hz, starting {LIU['window_start_after_trigger_s']}s after trigger")


[2026-06-14 11:09:42] Found 50 .mat files under /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_data/liu2024_figshare/sourcedata
[2026-06-14 11:09:49] Loaded 50 subjects. Example raw shape: (40, 33, 4000)
[2026-06-14 11:09:49] Window: 2000 samples = 4.0s @ 500 Hz, starting 0.0s after trigger


## 3.1 Precompute per-band covariances per subject (carried)

In [18]:
SUBJECT_COVS = {}     # sid -> {band_label: (n_trials, 29, 29)}
SUBJECT_LABELS = {}   # sid -> y (0/1)
trigger_report = []
band_labels = [f"{lo}-{hi}" for lo, hi in LIU["freq_bands_hz"]]

for sid in SUBJECTS:
    raw, y = RAW_SUBJECTS[sid]
    windows, detected, n_tr = subject_band_windows(raw)
    SUBJECT_COVS[sid] = {b: covariances(W) for b, W in windows.items()}
    SUBJECT_LABELS[sid] = y
    trigger_report.append({"subject_id": sid, "n_trials": n_tr,
                           "trigger_detected": detected, "fallback_used": n_tr - detected,
                           "class_counts": np.bincount(y, minlength=TARGET_N_CLASSES).tolist()})
trig_df = pd.DataFrame(trigger_report)
print(f"Trigger detection: {trig_df['trigger_detected'].sum()}/{trig_df['n_trials'].sum()} trials "
      f"({trig_df['fallback_used'].sum()} used the fixed-cue fallback).")
if trig_df["fallback_used"].sum() > 0:
    print("  NOTE: some trials had no marker==2; verify the marker channel / trigger value in CONFIG['liu'].")
trig_df.to_csv(ARTIFACT_DIR / "trigger_detection_report.csv", index=False)


[2026-06-14 11:11:05] Trigger detection: 2000/2000 trials (0 used the fixed-cue fallback).


## 3.2 Precompute log-covariances (for the fast permutation screen)

Matrix logs depend only on the covariances, not on labels, so we compute them once. The whole
permutation test then reduces to fast array math (class means + Frobenius distances).

In [19]:
def _reg_spd(C, shrink=1e-3):
    n = C.shape[0]; tr = np.trace(C) / n
    return (1 - shrink) * C + shrink * tr * np.eye(n)

SUBJECT_LOGS = {}
for sid in SUBJECTS:
    SUBJECT_LOGS[sid] = {b: np.stack([_logm_spd(_reg_spd(C)) for C in covs])
                         for b, covs in SUBJECT_COVS[sid].items()}
print(f"Precomputed log-covariances for {len(SUBJECT_LOGS)} subjects x {len(band_labels)} bands.")


[2026-06-14 11:11:07] Precomputed log-covariances for 50 subjects x 8 bands.


# 4. Permutation-test machinery

`cv_balacc_on_splits` scores the log-Euclidean nearest-mean classifier on fixed CV splits;
`perm_pvalue` shuffles labels (splits held fixed, so only the label–feature association is
broken) and returns a one-sided p-value; `bh_fdr` is Benjamini–Hochberg.

In [20]:
import scipy.stats as stats
import warnings
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
DEC = CONFIG["decode"]
BB = DEC["broadband"]
assert BB in band_labels, f"broadband {BB} not in carried bands {band_labels}"

def cv_balacc_on_splits(logs, y, splits, n_classes=TARGET_N_CLASSES):
    scores = []
    for tr, te in splits:
        ytr = y[tr]
        means = np.stack([logs[tr][ytr == c].mean(0) if np.any(ytr == c)
                          else np.zeros_like(logs[0]) for c in range(n_classes)])
        d = np.stack([np.linalg.norm(logs[te] - m, axis=(1, 2)) for m in means], axis=1)
        scores.append(balanced_accuracy_score(y[te], d.argmin(1)))
    return float(np.mean(scores))

def make_splits(y, folds, repeats, seed):
    sp = []
    for r in range(repeats):
        skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed + r)
        sp += list(skf.split(np.arange(len(y)), y))
    return sp

def perm_pvalue(stat_fn, n_perm, seed):
    obs = stat_fn(None)
    rng = np.random.default_rng(seed)
    null = np.array([stat_fn(rng) for _ in range(n_perm)])
    p = (1.0 + np.sum(null >= obs - 1e-12)) / (n_perm + 1.0)
    return obs, float(p), null

def bh_fdr(pvals):
    p = np.asarray(pvals, float); n = len(p); order = np.argsort(p); ranked = p[order]
    q = np.minimum.accumulate((ranked * n / np.arange(1, n + 1))[::-1])[::-1]
    out = np.empty(n); out[order] = np.clip(q, 0, 1)
    return out


# 5. Per-subject decodability test

For each subject: honest broadband test statistic + permutation p-value; the affine-invariant
FgMDM accuracy (reported, no permutation); and the selection-aware oracle statistic with its own
corrected p-value.

In [21]:
rows = []
subjects = SUBJECTS[:DEC["max_subjects"]] if DEC["max_subjects"] else SUBJECTS
for sid in subjects:
    y = SUBJECT_LABELS[sid]
    splits = make_splits(y, DEC["cv_folds"], DEC["cv_repeats"], DEC["seed"])
    bb_logs = SUBJECT_LOGS[sid][BB]

    def honest(rng, bb_logs=bb_logs, y=y, splits=splits):
        yy = y if rng is None else rng.permutation(y)
        return cv_balacc_on_splits(bb_logs, yy, splits)
    obs_h, p_h, _ = perm_pvalue(honest, DEC["n_permutations"], DEC["seed"] + int(sid))
    row = {"subject_id": sid, "honest_balacc_8_30": obs_h, "p_honest": p_h}

    if DEC["oracle_selection_null"]:
        logs_by_band = [SUBJECT_LOGS[sid][b] for b in band_labels]
        def oracle(rng, logs_by_band=logs_by_band, y=y, splits=splits):
            yy = y if rng is None else rng.permutation(y)
            return max(cv_balacc_on_splits(L, yy, splits) for L in logs_by_band)
        obs_o, p_o, _ = perm_pvalue(oracle, DEC["n_permutations"], DEC["seed"] + 1000 + int(sid))
        row["oracle_balacc_maxband"] = obs_o
        row["p_oracle_selaware"] = p_o

    if DEC["report_fgmdm"] and HAVE_PYRIEMANN:
        C = SUBJECT_COVS[sid][BB]; sc = []
        for tr, te in splits:
            try:
                clf = FgMDM(metric=LIU["metric"]).fit(C[tr], y[tr])
                sc.append(balanced_accuracy_score(y[te], clf.predict(C[te])))
            except Exception:
                pass
        row["fgmdm_balacc_8_30"] = float(np.mean(sc)) if sc else None

    rows.append(row)
    print(f"  subject {sid}: honest(8-30)={obs_h:.3f} p={p_h:.3f}"
          + (f" | oracle={row.get('oracle_balacc_maxband', float('nan')):.3f} p_sa={row.get('p_oracle_selaware', float('nan')):.3f}" if DEC["oracle_selection_null"] else "")
          + (f" | FgMDM={row['fgmdm_balacc_8_30']:.3f}" if row.get("fgmdm_balacc_8_30") is not None else ""))

DECODE_DF = pd.DataFrame(rows)
DECODE_DF["q_honest_fdr"] = bh_fdr(DECODE_DF["p_honest"])
DECODE_DF["bonferroni_honest"] = np.clip(DECODE_DF["p_honest"] * len(DECODE_DF), 0, 1)
DECODE_DF["decodable"] = DECODE_DF["q_honest_fdr"] < DEC["alpha"]
DECODE_DF = DECODE_DF.sort_values("honest_balacc_8_30", ascending=False).reset_index(drop=True)
DECODE_DF.to_csv(ARTIFACT_DIR / "decodability_per_subject.csv", index=False)


[2026-06-14 11:11:43]   subject 1: honest(8-30)=0.500 p=0.478 | oracle=0.550 p_sa=0.583 | FgMDM=0.450


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known

[2026-06-14 11:12:20]   subject 2: honest(8-30)=0.338 p=0.950 | oracle=0.425 p_sa=0.968 | FgMDM=0.463


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known

[2026-06-14 11:12:56]   subject 3: honest(8-30)=0.512 p=0.464 | oracle=0.575 p_sa=0.438 | FgMDM=0.512


...

[2026-06-14 11:36:40]   subject 42: honest(8-30)=0.450 p=0.717 | oracle=0.512 p_sa=0.816 | FgMDM=0.588
[2026-06-14 11:37:16]   subject 43: honest(8-30)=0.525 p=0.403 | oracle=0.525 p_sa=0.656 | FgMDM=0.487


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known

[2026-06-14 11:37:53]   subject 44: honest(8-30)=0.463 p=0.663 | oracle=0.688 p_sa=0.025 | FgMDM=0.750
[2026-06-14 11:38:30]   subject 45: honest(8-30)=0.463 p=0.652 | oracle=0.613 p_sa=0.266 | FgMDM=0.588
[2026-06-14 11:39:08]   subject 46: honest(8-30)=0.312 p=0.972 | oracle=0.487 p_sa=0.852 | FgMDM=0.438
[2026-06-14 11:39:46]   subject 47: honest(8-30)=0.525 p=0.401 | oracle=0.562 p_sa=0.571 | FgMDM=0.463
[2026-06-14 11:40:26]   subject 48: honest(8-30)=0.412 p=0.842 | oracle=0.537 p_sa=0.710 | FgMDM=0.537
[2026-06-14 11:41:02]   subject 49: honest(8-30)=0.512 p=0.461 | oracle=0.725 p_sa=0.032 | FgMDM=0.588


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known

[2026-06-14 11:41:38]   subject 50: honest(8-30)=0.375 p=0.899 | oracle=0.487 p_sa=0.829 | FgMDM=0.575


# 6. Decodable subgroup and full-cohort comparison

In [22]:
dec = DECODE_DF[DECODE_DF["decodable"]]
print(f"Decodable subjects (BH-FDR q<{DEC['alpha']}): {dec['subject_id'].tolist()}")
print(f"  n decodable = {len(dec)} / {len(DECODE_DF)}")
print(f"  also significant under Bonferroni: "
      f"{DECODE_DF[DECODE_DF['bonferroni_honest']<DEC['alpha']]['subject_id'].tolist()}")

def grp(df, col):
    v = df[col].dropna()
    return (float(v.mean()), float(v.std())) if len(v) else (None, None)

summary = {"n_subjects": int(len(DECODE_DF)), "n_decodable": int(len(dec)),
           "decodable_subjects": [int(s) for s in dec["subject_id"].tolist()],
           "alpha": DEC["alpha"], "n_permutations": DEC["n_permutations"], "broadband": BB}
print("\n================ honest balanced accuracy (8-30 Hz, no band selection) ================")
for label, sub in [("full cohort", DECODE_DF), ("decodable subgroup", dec)]:
    m, s = grp(sub, "honest_balacc_8_30")
    print(f"  {label:<20}: logE-MDM {100*m:.1f}% (sd {100*s:.1f})" if m is not None else f"  {label}: n/a")
    summary[f"honest_{label.replace(' ','_')}"] = m
    if "fgmdm_balacc_8_30" in DECODE_DF.columns:
        mf, sf = grp(sub, "fgmdm_balacc_8_30")
        if mf is not None:
            print(f"  {'':<20}  FgMDM    {100*mf:.1f}% (sd {100*sf:.1f})")
            summary[f"fgmdm_{label.replace(' ','_')}"] = mf

if "oracle_balacc_maxband" in DECODE_DF.columns:
    print("\n================ oracle best-of-8-bands (selection-aware p-values) ================")
    n_or_sig = int((DECODE_DF["p_oracle_selaware"] < DEC["alpha"]).sum())
    mo, so = grp(DECODE_DF, "oracle_balacc_maxband")
    print(f"  full cohort oracle accuracy: {100*mo:.1f}% (sd {100*so:.1f})")
    print(f"  subjects whose oracle accuracy is significant under the SELECTION-AWARE null: "
          f"{n_or_sig}/{len(DECODE_DF)}")
    print(f"  -> the gap between this oracle % and the honest % is the selection-on-noise inflation.")
    summary["oracle_full_cohort"] = mo
    summary["n_oracle_significant_selaware"] = n_or_sig

with open(ARTIFACT_DIR / "decodability_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

try:
    from IPython.display import display
    display(DECODE_DF.round(3))
except Exception:
    print(DECODE_DF.round(3).to_string(index=False))


[2026-06-14 11:41:38] Decodable subjects (BH-FDR q<0.05): ['40', '23']
[2026-06-14 11:41:38]   n decodable = 2 / 50
[2026-06-14 11:41:38]   also significant under Bonferroni: ['23']

[2026-06-14 11:41:38] ================ honest balanced accuracy (8-30 Hz, no band selection) ================
[2026-06-14 11:41:38]   full cohort         : logE-MDM 48.1% (sd 11.5)
[2026-06-14 11:41:38]                         FgMDM    53.4% (sd 11.1)
[2026-06-14 11:41:38]   decodable subgroup  : logE-MDM 75.0% (sd 1.8)
[2026-06-14 11:41:38]                         FgMDM    75.6% (sd 0.9)

[2026-06-14 11:41:38] ================ oracle best-of-8-bands (selection-aware p-values) ================
[2026-06-14 11:41:38]   full cohort oracle accuracy: 56.8% (sd 10.6)
[2026-06-14 11:41:38]   subjects whose oracle accuracy is significant under the SELECTION-AWARE null: 9/50
[2026-06-14 11:41:38]   -> the gap between this oracle % and the honest % is the selection-on-noise inflation.


,subject_id,honest_balacc_8_30,p_honest,oracle_balacc_maxband,p_oracle_selaware,fgmdm_balacc_8_30,q_honest_fdr,bonferroni_honest,decodable
0,40,0.762,0.002,0.800,0.002,0.750,0.050,0.100,True
1,23,0.738,0.001,0.738,0.006,0.762,0.050,0.050,True
2,28,0.725,0.005,0.762,0.005,0.712,0.075,0.250,False
3,7,0.712,0.006,0.775,0.002,0.788,0.075,0.300,False
4,22,0.688,0.018,0.775,0.004,0.675,0.180,0.899,False
5,20,0.625,0.102,0.662,0.086,0.662,0.728,1.000,False
6,30,0.600,0.155,0.675,0.091,0.662,0.860,1.000,False
7,38,0.600,0.141,0.638,0.265,0.575,0.860,1.000,False
8,11,0.600,0.099,0.675,0.041,0.625,0.728,1.000,False
9,5,0.562,0.265,0.562,0.770,0.588,0.988,1.000,False


# 7. (Optional) S-JEPA vs Riemannian on the decodable subgroup

If `CONFIG["decode"]["sjepa_subject_metrics_path"]` points to a prior S-JEPA run's per-subject
balanced accuracy (CSV with `subject_id,balanced_accuracy`, or a JSON dict
`{subject_id: {"mean_balanced_accuracy": ...}}`), this aligns it to the decodable subgroup and
runs a paired test against the Riemannian baseline — the comparison that actually matters.

In [23]:
def load_sjepa_subject_metrics(path):
    path = Path(path)
    if not path.exists():
        print(f"S-JEPA metrics file not found: {path}"); return None
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        col = next((c for c in df.columns if "balanced" in c.lower()), None)
        sidc = next((c for c in df.columns if "subject" in c.lower()), df.columns[0])
        return {str(int(r[sidc])): float(r[col]) for _, r in df.iterrows() if col}
    data = json.load(open(path))
    out = {}
    for k, v in data.items():
        if isinstance(v, dict):
            val = v.get("mean_balanced_accuracy", v.get("balanced_accuracy"))
        else:
            val = v
        if val is not None:
            out[str(k)] = float(val)
    return out

sjepa_path = DEC.get("sjepa_subject_metrics_path")
if sjepa_path:
    sj = load_sjepa_subject_metrics(sjepa_path)
    if sj:
        ref_col = "fgmdm_balacc_8_30" if "fgmdm_balacc_8_30" in DECODE_DF.columns else "honest_balacc_8_30"
        comp = DECODE_DF[DECODE_DF["decodable"]].copy()
        comp["sjepa"] = comp["subject_id"].astype(str).map(sj)
        comp = comp.dropna(subset=["sjepa", ref_col])
        if len(comp) >= 2:
            print(f"Decodable subgroup (n={len(comp)}): Riemannian ({ref_col}) vs S-JEPA")
            print(comp[["subject_id", ref_col, "sjepa"]].round(3).to_string(index=False))
            print(f"\n  mean Riemannian {100*comp[ref_col].mean():.1f}%  |  mean S-JEPA {100*comp['sjepa'].mean():.1f}%")
            try:
                w = stats.wilcoxon(comp[ref_col], comp["sjepa"])
                print(f"  paired Wilcoxon: statistic={w.statistic:.1f}, p={w.pvalue:.4f}")
            except Exception as exc:
                print(f"  (Wilcoxon failed: {exc})")
            comp.to_csv(ARTIFACT_DIR / "subgroup_sjepa_vs_riemannian.csv", index=False)
        else:
            print("Not enough overlapping decodable subjects to compare.")
    else:
        print("Could not parse S-JEPA metrics.")
else:
    print("No S-JEPA metrics path set -> skipping subgroup comparison. "
          "Set CONFIG['decode']['sjepa_subject_metrics_path'] to a prior run's per-subject file.")


[2026-06-14 11:41:38] No S-JEPA metrics path set -> skipping subgroup comparison. Set CONFIG['decode']['sjepa_subject_metrics_path'] to a prior run's per-subject file.


# 8. Plots

In [24]:
if HAVE_MPL and not DECODE_DF.empty:
    d = DECODE_DF.sort_values("honest_balacc_8_30").reset_index(drop=True)
    x = np.arange(len(d))
    colors = ["#2c7fb8" if ok else "#bdbdbd" for ok in d["decodable"]]
    fig, ax = plt.subplots(figsize=(max(9, 0.28 * len(d)), 4.5))
    ax.bar(x, d["honest_balacc_8_30"], color=colors)
    ax.axhline(0.5, ls="--", c="red", lw=1, label="chance")
    ax.set_xticks(x); ax.set_xticklabels(d["subject_id"], rotation=90, fontsize=6)
    ax.set_ylabel("honest balanced accuracy (8-30 Hz)"); ax.set_ylim(0, 1)
    ax.set_title(f"Per-subject decodability (blue = significant, BH-FDR q<{DEC['alpha']})")
    ax.legend(); fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / "decodability_per_subject.png", dpi=160); plt.close(fig)
    print(f"Saved: {ARTIFACT_DIR/'decodability_per_subject.png'}")

    if "oracle_balacc_maxband" in DECODE_DF.columns:
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(DECODE_DF["honest_balacc_8_30"], DECODE_DF["oracle_balacc_maxband"],
                   c=["#2c7fb8" if ok else "#bdbdbd" for ok in DECODE_DF["decodable"]])
        lim = [0.2, 0.95]; ax.plot(lim, lim, "k--", lw=1, label="y = x")
        ax.axhline(0.5, ls=":", c="red", lw=1); ax.axvline(0.5, ls=":", c="red", lw=1)
        ax.set_xlabel("honest 8-30 Hz balanced accuracy")
        ax.set_ylabel("oracle best-of-8-bands accuracy")
        ax.set_title("Oracle vs honest (vertical gap = selection-on-noise inflation)")
        ax.legend(); fig.tight_layout()
        fig.savefig(ARTIFACT_DIR / "oracle_vs_honest.png", dpi=160); plt.close(fig)
        print(f"Saved: {ARTIFACT_DIR/'oracle_vs_honest.png'}")
else:
    print("Plots skipped (need matplotlib + results).")


[2026-06-14 11:41:38] Saved: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-decodability-permutation/20260614_1109_9d3f1b1b/decodability_per_subject.png
[2026-06-14 11:41:38] Saved: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-decodability-permutation/20260614_1109_9d3f1b1b/oracle_vs_honest.png


## 9. Save run metadata

In [25]:
run_metadata = {
    "run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR),
    "experiment_name": CONFIG["experiment_name"], "config_note": CONFIG["config_note"],
    "decode_config": DEC, "liu_config": LIU,
    "honest_classifier": "log-Euclidean nearest-Riemannian-mean (fixed 8-30 Hz)",
    "reported_classifier": "pyriemann.FgMDM" if HAVE_PYRIEMANN else "n/a",
    "summary": summary,
}
with open(ARTIFACT_DIR / "run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2, default=str)
print(f"Saved run metadata to: {ARTIFACT_DIR/'run_metadata.json'}")
for p in sorted(ARTIFACT_DIR.glob("*")):
    print(f"  - {p.name}")
try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass


[2026-06-14 11:41:38] Saved run metadata to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-decodability-permutation/20260614_1109_9d3f1b1b/run_metadata.json
[2026-06-14 11:41:38]   - config.json
[2026-06-14 11:41:38]   - decodability_per_subject.csv
[2026-06-14 11:41:38]   - decodability_per_subject.png
[2026-06-14 11:41:38]   - decodability_summary.json
[2026-06-14 11:41:38]   - oracle_vs_honest.png
[2026-06-14 11:41:38]   - run.log
[2026-06-14 11:41:38]   - run_metadata.json
[2026-06-14 11:41:38]   - trigger_detection_report.csv


# 10. How to use this

- **The decodable subgroup is your working cohort.** Subjects flagged `decodable` (BH-FDR
  q < α) are the ones with statistically real left-vs-right MI. Run every S-JEPA-vs-Riemannian
  comparison on this subset; on the full cohort both methods are at chance and any difference is
  noise.
- **Honest vs oracle.** The honest 8–30 Hz number is the defensible per-subject accuracy. The
  oracle best-of-8-bands number is higher, but the **selection-aware p-values** show how few
  subjects' oracle accuracy survives once the band-selection is in the null — that gap is the
  inflation behind a headline like "72%". Quote the honest number when comparing to S-JEPA.
- **FgMDM vs log-Euclidean.** The permutation screen uses the fast log-Euclidean classifier
  (a valid decodability detector); the affine-invariant FgMDM column is the paper's actual
  classifier and should track it closely on the decodable subjects.
- **Next steps.** (1) If S-JEPA loses to FgMDM even on the decodable subgroup, the deep model is
  not earning its complexity in the within-subject 40-trial regime — the clean result for your
  advisor. (2) The path to give S-JEPA a fair chance is cross-subject Riemannian alignment +
  pooling with leave-subject-out, evaluated on this same decodable subgroup.

**Caveats.** Decodability depends on the fixed 0–4 s window and 8–30 Hz band; a subject flagged
non-decodable here might still decode in a different time window (use the shifting-window
notebook to check). The permutation null assumes exchangeable trials within subject, which holds
for these balanced, single-session recordings.